In [ ]:
!pip install -q --no-warn-conflicts google-adk google-auth==2.43.0

In [ ]:
!pip install -q -U beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.5/236.5 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.48.0 which is incompatible.


In [ ]:
import datetime
import json
import smtplib
from pathlib import Path
from email.message import EmailMessage
from google.colab import userdata
from google import genai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- ADK CONFIGURATION ---

# 1. Persistence & Multi-Judge Settings
STATE_FILE = "/content/drive/MyDrive/legal_project/governance_state.json"
JUDGE_COUNT = 3
MAJORITY_THRESHOLD = 2

# 2. Hierarchical Topic Taxonomy
# This informs the Classifier Agent's logic
TOPIC_HIERARCHY = {
    "Economic": ["Federal Budget", "Taxation", "Trade"],
    "Governance": ["Agency Regulation", "Executive Orders", "Jurisdiction"],
    "Social": ["Civil Rights", "Health Policy", "Education"],
    "Security": ["Defense", "Cybersecurity", "Immigration"]
}

# 3. Enhanced Audit Rubric (Simplified for 3-Judge Majority)
# This is the "Truth Standard" for the Interpretations
INTERPRETATION_RUBRIC = '''
1. OBJECTIVITY: Zero partisan adjectives or emotional framing.
2. ACCESSIBILITY: Written for a 6th-8th grade reading level (plain language).
3. FIDELITY: Captures the primary legal mechanism without omission.
VERDICT: PASS or FAIL.
'''

In [ ]:
# --- GLOBAL SESSION SERVICE (Maintains memory across runs) ---

session_service = InMemorySessionService()

In [ ]:
# --- AGENT DEFINITIONS (Cost-Optimized) ---

# Core Processing Agents
classifier_agent = Agent(
    name="Law_Classifier",
    model_id="gemini-2.5-flash",
    system_instruction=f"Categorize laws into: {TOPIC_HIERARCHY}. Output tags only."
)

legal_analyst_agent = Agent(
    name="Legal_Analyst",
    model_id="gemini-2.5-flash",
    system_instruction="Create a technical summary and a neutral plain-language interpretation."
)

# High-Precision Consensus Judges
judge_1 = Agent(name="Judge_Gemini", model_id="gemini-3-pro", system_instruction=RUBRIC)
judge_2 = Agent(name="Judge_Grok", model_id="grok-4.1-thinking", system_instruction=RUBRIC)
judge_3 = Agent(name="Judge_GPT", model_id="gpt-5.2-high", system_instruction=RUBRIC)

# Management
orchestrator_agent = Agent(
    name="Legal_Orchestrator",
    model_id="gemini-2.5-flash",
    system_instruction="Coordinate the pipeline and determine final status by majority vote."
)

ValidationError: 2 validation errors for LlmAgent
model_id
  Extra inputs are not permitted [type=extra_forbidden, input_value='gemini-2.5-flash', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden
system_instruction
  Extra inputs are not permitted [type=extra_forbidden, input_value="Categorize laws into: {'...n']}. Output tags only.", input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [ ]:
# --- GOVERNANCE & FAILSAFE FUNCTIONS ---

def check_failsafe_trigger():
    """Checks if the 10-day admin window has expired."""
    if not Path(STATE_FILE).exists():
        return "READY"

    with open(STATE_FILE, 'r') as f:
        state = json.load(f)

    deadline = datetime.datetime.fromisoformat(state['deadline_date'])

    if datetime.datetime.now() > deadline:
        return "TRIGGER_AUTO_JURY"
    return "WAITING"

def save_discrepancy_state(admin_email, interpretation_text):
    """Persists state to Google Drive to handle the 10-day countdown."""
    deadline = datetime.datetime.now() + datetime.timedelta(days=10)
    state_data = {
        "status": "AWAITING_ADMIN_OVERRIDE",
        "deadline_date": deadline.isoformat(),
        "admin_contact": admin_email,
        "interpretation_to_review": interpretation_text
    }
    with open(STATE_FILE, 'w') as f:
        json.dump(state_data, f)
    print(f"Governance State Locked. Deadline: {deadline}")

# --- THE EXECUTIVE RUNNER ---

def run_adk_governance_engine(url, doc_date, admin_email):
    print("--- ADK GOVERNANCE ENGINE STARTING ---")

    # 1. Check for Active Failsafes
    status = check_failsafe_trigger()
    if status == "TRIGGER_AUTO_JURY":
        print("CRITICAL: 10-day window expired. Triggering Ensemble...")
        # (Implementation of execute_jury_ensemble would follow here)
        return "Jury of Five Verdict: PASS (Automatic)"

    # 2. Initialize ADK Runner for overall context and session management.
    # The orchestrator_agent is set as the default agent for the runner.
    adk_runner = Runner(
        app_name="Legal_Governance_v2",
        agent=orchestrator_agent,
        session_service=session_service
    )

    # 3. Execution Pipeline - Use adk_runner.run() for all agent interactions
    print(f"Analyzing: {url}")

    # The orchestrator_agent processes the document date
    # Explicitly specify orchestrator_agent as argument to adk_runner.run()
    adk_runner.run(doc_date, agent=orchestrator_agent)

    # Subsequent agents are called via adk_runner, explicitly specifying the agent
    raw_data = adk_runner.run(url, agent=rag_agent)
    facts = adk_runner.run(raw_data, agent=analyst_agent)
    interpretation = adk_runner.run(facts, agent=neutrality_agent)

    # 4. Parallel Audit
    print("Initiating Dual-Judge Audit...")
    audit_1 = adk_runner.run(interpretation, agent=judge_gpt)
    audit_2 = adk_runner.run(interpretation, agent=judge_claude)

    # 5. Verdict Branching
    verdict = adk_runner.run({"gpt": audit_1, "claude": audit_2}, agent=consensus_foreman)

    if "VERIFIED" in verdict:
        print("SUCCESS: Neutrality Confirmed.")
        return interpretation
    else:
        save_discrepancy_state(admin_email, interpretation)
        # send_admin_alert(verdict, "10-Day Window Initiated")
        return "System Halted: Discrepancy detected. Admin notified."

# --- TEST EXECUTION ---
TEST_URL = "https://www.whitehouse.gov/briefing-room/presidential-actions/2025/12/executive-order-federal-ai-preemption/"
print(run_adk_governance_engine(TEST_URL, "2025-12-15", "admin@example.com"))